# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

## Get Started With Requests

In [11]:
import requests

## Make a GET Request

In [12]:
response = requests.get("https://api.github.com")

print(response)

<Response [200]>


## Inspect the Response

### Status Codes

In [13]:
print(response.status_code)

if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Not Found.")

200
Success!


### Response Content

In [14]:
print(response.content)
print(response.text)
print(response.json())
print(type(response.content))
print(type(response.text))
print(type(response.json()))

response.encoding = "utf-8"
print(response.text)

response_dict = response.json()
print(response_dict["emojis_url"])

b'{"current_user_url":"https://api.github.com/user","current_user_authorizations_html_url":"https://github.com/settings/connections/applications{/client_id}","authorizations_url":"https://api.github.com/authorizations","code_search_url":"https://api.github.com/search/code?q={query}{&page,per_page,sort,order}","commit_search_url":"https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}","emails_url":"https://api.github.com/user/emails","emojis_url":"https://api.github.com/emojis","events_url":"https://api.github.com/events","feeds_url":"https://api.github.com/feeds","followers_url":"https://api.github.com/user/followers","following_url":"https://api.github.com/user/following{/target}","gists_url":"https://api.github.com/gists{/gist_id}","hub_url":"https://api.github.com/hub","issue_search_url":"https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}","issues_url":"https://api.github.com/issues","keys_url":"https://api.github.com/user/keys","label_sea

### Response Headers

In [15]:
print(response.headers)
print(response.headers["Content-Type"])
print(response.headers["content-type"])

{'Date': 'Wed, 16 Sep 2026 01:50:28 GMT', 'Cache-Control': 'public, max-age=60, s-maxage=60', 'Vary': 'Accept,Accept-Encoding, Accept, X-Requested-With', 'ETag': '"4f825cc84e1c733059d46e76e6df9db557ae5254f9625dfe8e1b09499c449438"', 'x-github-api-version-selected': '2022-11-28', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset, Warning', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Referrer-Policy': 'origin-when-cross-origin, strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'none'", 'Server': 'github.com', 'Content-Type': 'application/json; ch

## Query String Parameters

In [16]:
params = {"q": "language:python",
"sort": "stars",
"order": "desc"}

response = requests.get(
    "https://api.github.com/search/repositories",
    params=params)

print(response.status_code)

print(response.url)

json_response = response.json()

repositories = json_response["items"]

for repo in repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

200
https://api.github.com/search/repositories?q=language%3Apython&sort=stars&order=desc
Name: public-apis
Description: A collective list of free APIs
Stars: 480631

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396862

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 370220



## Customize Request Headers

In [17]:
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"}
)

json_response = response.json()

first_repository = json_response["items"][0]

print(first_repository["text_matches"][0]["matches"])

[{'text': 'Real Python', 'indices': [23, 34]}]


## Improve Performance

### Timeouts

In [18]:
response = requests.get("https://api.github.com",
timeout=1)

print(response.status_code)

response = requests.get("https://api.github.com",
timeout=(3.05, 5))

print(response.status_code)

from requests.exceptions import Timeout

try:
    response = requests.get(
        "https://api.github.com",
        timeout=(3.05, 5)
    )
except Timeout:
    print("The request timed out")
else:
    print("The request did not time out")

200
200
The request did not time out


### Sessions

In [19]:
session = requests.Session()

response1 = session.get("https://api.github.com")
response2 = session.get("https://api.github.com/events")

print(response1.status_code)
print(response2.status_code)

session.close()

200
200


### Retries

In [20]:
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=2,
    status_forcelist=[429, 500, 502, 503, 504]
)

github_adapter = HTTPAdapter(max_retries=retry_strategy)

with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)

    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")